In [1]:
!pip install pytabkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 12.5 MB/s eta 0:00:00


In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from catboost import Pool
from sklearn.base import clone, BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import TargetEncoder, LabelEncoder, StandardScaler
from sklearn.utils.validation import check_is_fitted
from pytabkit import RealMLP_TD_Classifier, TabM_D_Classifier
from itertools import combinations
from sklearn.metrics import roc_auc_score
from typing import List, Union, Optional
import copy
import warnings
warnings.filterwarnings('ignore')
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s6e2/sample_submission.csv
/kaggle/input/playground-series-s6e2/train.csv
/kaggle/input/playground-series-s6e2/test.csv


In [3]:
class config:
    SEED = 42
    N_FOLDS = 5
    TARGET = 'Heart Disease'
    
    INPUT_DIR = '/kaggle/input/playground-series-s6e2'

class_mapping = {
    'Presence': 1,
    'Absence': 0
}
rev_class_mapping = {
    0: 'Absence',
    1: 'Presence'
}

CONFIG = config()

In [4]:
train = pd.read_csv(f'{CONFIG.INPUT_DIR}/train.csv')
train['source'] = 'train'
test = pd.read_csv(f'{CONFIG.INPUT_DIR}/test.csv')
test['source'] = 'test'
sample_sub = pd.read_csv(f'{CONFIG.INPUT_DIR}/sample_submission.csv')

# org = pd.read_csv('/kaggle/input/heartdisease/Heart_Disease_Prediction.csv')
# org['source'] = 'original'
# strat_feature

In [5]:
NUMS = [col for col in train.columns if col not in ['id', 'Heart Disease', 'source']]
# NUMS = [col for col in NUMS if col not in BINS]
HIGH_CARDINALITY = [col for col in NUMS if train[col].nunique() > 40]

In [6]:
# BINS = []
# for q in [5]:
#     for c in HIGH_CARDINALITY:
#         n = f'{c}_{q}_bin'
#         train_bins, bins = pd.qcut(train[c], q=q, labels=False, retbins=True, duplicates='drop')
#         train[n] = train_bins
#         test[n] = pd.cut(test[c], bins=bins, labels=False, include_lowest=True)
#         BINS.append(n)

# print(BINS)
# print('='*30)
# print(len(BINS))

In [7]:
combine = pd.concat([train.drop(columns='id'), test.drop(columns='id')], ignore_index=True).reset_index()

In [8]:
BASE_FEATURES = ['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol',
       'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina',
       'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium']

In [9]:
def feature_engineering(df):
    BP_THRESHOLD = 140
    CHOL_THRESHOLD = 240

    df = df.copy()
    
    df['elevated_bp'] = (df['BP'] >= BP_THRESHOLD).astype(int)
    df['elevated_chol'] = (df['Cholesterol'] >= CHOL_THRESHOLD).astype(int)

    # df['risk_factor_count'] = df['elevated_bp'] + df['elevated_chol']
    # df['risk_age'] = df['Age'] * df['risk_factor_count']
    return df

combine = feature_engineering(combine)

In [10]:
# for df, name in zip([train, test], ['train', 'test']):
#     print(f'NULL VALUE COUNTS FOR {name}:')
#     print(df.isnull().sum())
#     print('='*30)
#     print(f'{name} shape:')
#     print(df.shape)
#     print('='*30)
#     if name == 'train':
#         print('General EDA -- TRAIN ONLY', end='\n')
#         print(f'Dtypes :', end='\n')
#         print(df.dtypes)
#         print('='*30)
#         print(f'NUMBER OF UNIQUE VALUES :', end='\n')
#         print(df.nunique())
#         print('='*30)
    

In [11]:
print(NUMS)
print(HIGH_CARDINALITY)

['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium']
['Age', 'BP', 'Cholesterol', 'Max HR', 'ST depression']


In [12]:
# CATS = []
# for c in NUMS:
#     n = f'{c}_cat'
#     combine[n] = combine[c].astype(str).astype('category')
#     CATS.append(n)

# print(CATS)
# print('='*30)
# print(len(CATS))

In [13]:
# INTER = []
# for c1, c2 in combinations(CATS, 2):
#     n = f'{c1}_{c2}'
#     combine[n] = (combine[c1].astype(str) + '_' + combine[c2].astype(str)).astype('category')
#     INTER.append(n)

# print(INTER)
# print('='*30)
# print(len(INTER))

In [14]:
# INTER1 = []
# for c1, c2 in combinations(NUMS, 2):
#     n = f'{c1}_{c2}'
#     combine[n] = combine[c1] * combine[c2]
#     INTER1.append(n)

# print(INTER1)
# print('='*30)
# print(len(INTER1))

In [15]:
# ENC = []

# for c in INTER:
#     n = f'{c}_enc'
#     combine[n], _ = pd.factorize(combine[c])
#     ENC.append(n)

# print(ENC)
# print('='*30)
# print(len(ENC))

In [16]:
train = combine.loc[combine['source']=='train']
test = combine.loc[combine['source']=='test']
# org = combine.loc[combine['source']=='original']

In [17]:
# print(train.groupby('risk_factor_count')['Heart Disease'].mean())
# print(train.groupby('elevated_bp')['Heart Disease'].mean())

In [18]:
# org[CONFIG.TARGET] = org[CONFIG.TARGET].map(class_mapping)
# TE = []
# TE1 = []
# COND_TE = []

# # ALL_CATS = CATS + CATS1 + BINS
# GLOBAL_MEAN = org[config.TARGET].mean()
# # GLOBAL_STD = train_org_n[config.TARGET].std()
# GLOBAL_MEDIAN = org[config.TARGET].median()
# GLOBAL_COUNT = org[config.TARGET].count()

# ALPHA = 10 

# for c in CATS:
#     # if i%5==0: print(i, end='===')
#     for target in [config.TARGET]:
#         # if target=='study_hours':
#         #     if c in CATS:
#         #         tmp_mean = train_org_n.groupby(c)[target].mean()
#         #         tmp_median = train_org_n.groupby(c)[target].median()
#         #         tmp_std = train_org_n.groupby(c)[target].std()
#         #         tmp_min = train_org_n.groupby(c)[target].min()
#         #         tmp_max = train_org_n.groupby(c)[target].max()

#         # else:
#         tmp_mean = org.groupby(c)[target].mean()
#         tmp_count = org.groupby(c)[target].count()
#         tmp_median = org.groupby(c)[target].median()
#         tmp_std = org.groupby(c)[target].std()
#         tmp_skew = org.groupby(c)[target].skew()
#             # tmp_min = train_org_n.groupby(c)[target].min()
#             # tmp_max = train_org_n.groupby(c)[target].max()

        
#         # tmp_count = train_org_n.groupby(c)[target].size()
#         # tmp_mean_delta = tmp_mean - GLOBAL_MEAN
#         # tmp_std_delta = tmp_std - GLOBAL_STD
#         # tmp_mean_ratio = tmp_mean / GLOBAL_MEAN
#         # tmp_std_ratio = tmp_std / GLOBAL_STD
#         # tmp_count = train_org_n.groupby(c)[config.TARGET].size()

#         n_mean = f'TE_{c}_{target}_mean'
#         n_count = f'TE_{c}_{target}_count'
#         n_median = f'TE_{c}_{target}_median'
#         n_std = f'TE_{c}_{target}_std'
#         n_skew = f'TE_{c}_{target}_skew'
#         # n_min = f'TE_{c}_{target}_min'
#         # n_max = f'TE_{c}_{target}_max'
#         # n_count = f'TE_{c}_{target}_count'
#         # n_mean_delta = f'TE_{c}_{target}_mean_delta'
#         # n_std_delta = f'TE_{c}_{target}_std_delta'
#         # n_mean_ratio = f'TE_{c}_{target}_mean_ratio'
#         # n_std_ratio = f'TE_{c}_{target}_std_ratio'

#         # n_c = f'TE_{c}_count'
#         print(f'{n_mean}, {n_count}, {n_std}, {n_median}, {n_skew}', end=' ')
#         tmp_mean.name = n_mean
#         tmp_median.name = n_median
#         tmp_std.name = n_std
#         # tmp_min.name = n_min
#         # tmp_max.name = n_max
#         tmp_count.name = n_count
#         tmp_skew.name = n_skew
#         # tmp_mean_delta.name = n_mean_delta
#         # tmp_std_delta.name = n_std_delta
#         # tmp_mean_ratio.name = n_mean_ratio
#         # tmp_std_ratio.name = n_std_ratio
        
    
#         stats = (pd.concat([tmp_mean, tmp_count, tmp_std, tmp_median, tmp_skew], axis=1).reset_index().rename(columns={'index': c}))
#         org = org.merge(stats, on=c, how='left')
#         train = train.merge(stats, on=c, how='left')
#         test = test.merge(stats, on=c, how='left')

#         if target==config.TARGET:
#             TE.append(n_mean)
#             TE.append(n_median)
#             TE.append(n_std)
#             TE.append(n_skew)
#             # TE.append(n_min)
#             # TE.append(n_max)
#             TE.append(n_count)
#             # TE.append(n_mean_delta)
#             # TE.append(n_std_delta)
#             # TE.append(n_mean_ratio)
#             # TE.append(n_std_ratio)

#         else:
#             TE1.append(n_mean)
#             # TE.append(n_median)
#             # TE1.append(n_std)
#             # TE1.append(n_min)
#             # TE1.append(n_max)   
#     # TE.append(n_c)


In [19]:
# org[CONFIG.TARGET] = org[CONFIG.TARGET].map(class_mapping)
# def add_engineered_features(df):
#     df_temp = df.copy()
    
#     for col in NUMS: 
#         if col in org.columns:
           
#             stats = org.groupby(col)['Heart Disease'].agg(['mean', 'median', 'std', 'skew', 'count']).reset_index()
         
#             stats.columns = [col] + [f"orig_{col}_{s}" for s in ['mean', 'median', 'std', 'skew', 'count']]
     
#             df_temp = df_temp.merge(stats, on=col, how='left') 
 
#             fill_values = {
#                 f"orig_{col}_mean": org['Heart Disease'].mean(),
#                 f"orig_{col}_median": org['Heart Disease'].median(),
#                 f"orig_{col}_std": 0,
#                 f"orig_{col}_skew": 0,
#                 f"orig_{col}_count": 0
#             }
#             df_temp = df_temp.fillna(value=fill_values)
            
#     return df_temp

# org = add_engineered_features(org)
# train = add_engineered_features(train)
# test = add_engineered_features(test) 

In [20]:
for df in [train, test]:
    int_cols = df.select_dtypes(include=['int64']).columns.to_list()
    float_cols = df.select_dtypes(include=['float64']).columns.to_list()
    df[int_cols] = df[int_cols].astype('int32')
    df[float_cols] = df[float_cols].astype('float32')
    # df.drop(columns=CATS, inplace=True)

In [21]:
# for df in [org, train, test]:
#     for col in TE:
#         if '_mean' in col:
#             # Fill NaNs with the global target mean
#             df[col] = df[col].fillna(GLOBAL_MEAN)
#         elif '_median' in col:
#             # Fill NaNs with the global target median
#             df[col] = df[col].fillna(GLOBAL_MEDIAN)
#         else:
#             # Fill count, std, and skew with 0
#             df[col] = df[col].fillna(0)

In [22]:
def freq_encode(X, features, freq_encodings):
    """Frequency encoding for multiple features"""
    X_freq = X.copy()
    for c in features:
        X_freq[f'{c}_freq'] = X[c].map(freq_encodings[c]).astype('float32').fillna(0)
    return X_freq

def target_mean_encode(X, features, target_mean_dict, global_mean):
    """Target mean encoding for multiple features"""
    X_mean = X.copy()
    for c in features:
        X_mean[f'{c}_mean'] = X[c].map(target_mean_dict[c]).astype('float32').fillna(global_mean)
    return X_mean

def target_count_encode(X, features, target_count_dict, global_count):
    """Target count encoding for multiple features"""
    X_count = X.copy()
    for c in features:
        X_count[f'{c}_count'] = X[c].map(target_count_dict[c]).astype('float32').fillna(global_count)
    return X_count

def kbins_discretize(X, numeric_cols, n_bins=10):
    """Create binned versions of numeric features"""
    X_binned = X.copy()
    for col in numeric_cols:
        # Create bins
        bins = pd.qcut(X[col], q=n_bins, labels=False, duplicates='drop')
        X_binned[f'{col}_bin'] = bins.astype('category')
    return X_binned

In [23]:
FEATURES = [col for col in train.columns if col not in ['id', 'Heart Disease', 'source', 'index', 'strat_feature']]
# FEATURES = [col for col in FEATURES if col not in INTER]
print(FEATURES)
print(len(FEATURES))

X = train[FEATURES]
# X = X.fillna(0)
# X_org = org[FEATURES]

y = train[CONFIG.TARGET].map(class_mapping)
# y_org = org[CONFIG.TARGET]

X_test = test[FEATURES]
# X_test = X_test.fillna(0)

['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium', 'elevated_bp', 'elevated_chol']
15


In [24]:
xgb_params = {
    'n_estimators': 10000,
    'learning_rate': 0.005,
    'subsample': 0.8,
    # 'colsample_by_tree': 0.7,
    # 'sampling_method': 'gradient_based',
    # 'reg_alpha': 2.0,
    # 'reg_lambda': 4.0,
    'eval_metric': 'auc',
    'enable_categorical': True,
    'tree_method': 'hist',
    'device': 'cuda',
    'early_stopping_rounds': 100,
    'random_state': CONFIG.SEED
}

lgb_params = {
    'n_estimators': 10_000,
    'learning_rate': 0.005,
    'max_depth': 8,                 # same philosophy
    'num_leaves': 2 ** 8,           # typical rule: ≤ 2^max_depth
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    # 'reg_lambda': 4.0,
    'random_state': CONFIG.SEED,
    'early_stopping_rounds': 100,
    'eval_metric': 'auc',
    'n_jobs': -1,
    # 'verbose': 200,
    'verbosity':-1,
    # 'device': 'cuda'
    # 'cat_feature': CATS,            # pass categorical indices/names
}

cat_params = {
    'iterations': 10_000,          # same as n_estimators
    'learning_rate': 0.005,
    'depth': 8,                    # max_depth equivalent
    'subsample': 0.8,              # bagging
    'colsample_bylevel': 0.7,      # feature fraction per split
    'reg_lambda': 4.0,             # L2
    'random_seed': CONFIG.SEED,
    'early_stopping_rounds': 100,
    'eval_metric': 'AUC',
    'thread_count': -1,            # use all cores
    'verbose': 200,                # same as XGB verbose
    # 'verbosity':-1
    # 'cat_features': CATS,          # list of column names / indices
}

real_mlp_params = {
        'device': 'cpu',
        'random_state': 42,
        'verbosity': 2,
        'val_metric_name': '1-auc_ovo',
        'n_epochs': 60,
        'batch_size': 2048,
        'n_ens': 20,
        'use_early_stopping': True,
        'early_stopping_additive_patience': 20,
        'early_stopping_multiplicative_patience': 1,
        'act': "mish",
        'embedding_size': 16,
        'first_layer_lr_factor': 0.5962121993798933,
        'hidden_sizes': "rectangular",
        'hidden_width': 384,
        'lr': 0.01,
        'ls_eps': 0.0,
        'ls_eps_sched': "coslog4",
        'max_one_hot_cat_size': 18,
        'n_hidden_layers': 3,
        'p_drop': 0.1,
        'p_drop_sched': "flat_cos",
        'plr_hidden_1': 16,
        'plr_hidden_2': 8,
        'plr_lr_factor': 0.1151437622270563,
        'plr_sigma': 2.3316811282666916,
        'scale_lr_factor': 2.244801835541429,
        'sq_mom': 1.0 - 0.011834054955582318,
        'wd': 0.02369230879235962,
    }

real_mlp_params2 = {
        'device': 'cuda',
        'random_state': 42,
        'verbosity': 2,
        'n_epochs': 100,
        'batch_size': 1048, 
        'n_ens': 8, 
        'use_early_stopping': True,
        'early_stopping_additive_patience': 20,
        'early_stopping_multiplicative_patience': 1,
        'act': "mish",
        'embedding_size': 8,
        'first_layer_lr_factor': 0.5962121993798933,
        'hidden_sizes': "rectangular",
        'hidden_width': 384,
        'lr': 0.04, 
        'ls_eps': 0.011498317194338772,
        'ls_eps_sched': "coslog4",
        'max_one_hot_cat_size': 18,
        'n_hidden_layers': 4, 
        'p_drop': 0.07301419697186451,
        'p_drop_sched': "flat_cos",
        'plr_hidden_1': 16, 
        'plr_hidden_2': 8,
        'plr_lr_factor': 0.1151437622270563,
        'plr_sigma': 2.3316811282666916,
        'scale_lr_factor': 2.244801835541429,
        'sq_mom': 1.0 - 0.011834054955582318,
        'wd': 0.02369230879235962,
    }
peak_realmlp = {
        'device': 'cuda',
        'random_state': 42,
        'verbosity': 2,
        'n_epochs': 100,
        'batch_size': 256, 
        'n_ens': 8, 
        'use_early_stopping': True,
        'early_stopping_additive_patience': 20,
        'early_stopping_multiplicative_patience': 1,
        'act': "mish",
        'embedding_size': 8,
        'first_layer_lr_factor': 0.5962121993798933,
        'hidden_sizes': "rectangular",
        'hidden_width': 384,
        'lr': 0.01,
        'ls_eps': 0.0,
        'ls_eps_sched': "coslog4",
        'max_one_hot_cat_size': 18,
        'n_hidden_layers': 3,
        'p_drop': 0.1,
        'p_drop_sched': "flat_cos",
        'plr_hidden_1': 16,
        'plr_hidden_2': 8,
        'plr_lr_factor': 0.1151437622270563,
        'plr_sigma': 2.3316811282666916,
        'scale_lr_factor': 2.244801835541429,
        'sq_mom': 1.0 - 0.011834054955582318,
        'wd': 0.02369230879235962,
    } 

In [25]:
# X = X.iloc[:1000]
# y = y.iloc[:1000]
# X_test = X_test.iloc[:1000]

In [26]:
skf = StratifiedKFold(n_splits=CONFIG.N_FOLDS, random_state=CONFIG.SEED, shuffle=True)
kf = KFold(n_splits=CONFIG.N_FOLDS, random_state=CONFIG.SEED, shuffle=True)

oof_preds = np.zeros(len(X))
realmlp_oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
realmlp_test_preds = np.zeros(len(X_test))
strat_cols = ['Thallium', 'Chest pain type', 'Heart Disease']
le = LabelEncoder()
stratify_feature = le.fit_transform(train[strat_cols].astype(str).agg('_'.join, axis=1))
# stratify_feature = X['strat_feature']
pseudo_threshold = 0.5
for col in X.columns:
        X[col] = X[col].astype(str).astype('category')
        # X_org[col] = X_org[col].astype(str).astype('category')
        # X_val[col] = X_val[col].astype(str).astype('category')
        X_test[col] = X_test[col].astype(str).astype('category')

for i, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    X_test_n = X_test.copy()
    # X_org_n = X_org.copy()
    # y_org_n = y_org.copy()
    # X_org_n = pd.concat([X_org_n]*3, axis=0, ignore_index=True)
    # y_org_n = pd.concat([y_org_n]*3, axis=0, ignore_index=True)
    # X_train = pd.concat([X_train, X_org_n], axis=0, ignore_index=True)
    # y_train = pd.concat([y_train, y_org_n], axis=0, ignore_index=True)
    # model = clone(xgb.XGBClassifier(**xgb_params))
    # model = clone(cb.CatBoostClassifier(**cat_params))
    # model = clone(lgb.LGBMClassifier(**lgb_params))

    model = clone(RealMLP_TD_Classifier(**real_mlp_params2))
    
    # freq_encodings = {}
    # for c in CATS:
    #     # n = f'{c}_freq'
    #     freq_encodings[c] = X_train[c].value_counts(normalize=True).to_dict()
    # X_train = freq_encode(X_train, CATS, freq_encodings)
    # X_val = freq_encode(X_val, CATS, freq_encodings)
    # X_test_n = freq_encode(X_test_n, CATS, freq_encodings)
        

    for c in BASE_FEATURES:
        n = f'{c}_mean_te'
        TE = TargetEncoder(cv=5, random_state=42, shuffle=True)
        X_train[n] = TE.fit_transform(pd.DataFrame(X_train[c]), y_train).flatten()
        X_val[n] = TE.transform(pd.DataFrame(X_val[c])).flatten()
        X_test_n[n] = TE.transform(pd.DataFrame(X_test[c])).flatten()

    global_median = y_train.median()

    for c in BASE_FEATURES:
        # 1. Create a temporary training DF to calculate stats
        temp_train = pd.DataFrame({c: X_train[c], 'target': y_train})
        
        # 2. Calculate group statistics
        stats = temp_train.groupby(c)['target'].agg(['median', 'std', 'count']).reset_index()
        
        # Rename columns to avoid collisions
        stats.columns = [c, f'{c}_median_te', f'{c}_std_te', f'{c}_count_te']
        
        # 3. Merge onto all sets
        X_train = X_train.merge(stats, on=c, how='left')
        X_val = X_val.merge(stats, on=c, how='left')
        X_test_n = X_test_n.merge(stats, on=c, how='left')
        
        # 4. Handle NaNs with your specific rules
        # Mean/Median use global fold values, others use 0
        fill_rules = {
            # f'{c}_mean_te': global_mean,
            f'{c}_median_te': global_median,
            f'{c}_std_te': 0,
            f'{c}_count_te': 0,
            f'{c}_skew_te': 0
        }
        
        for df in [X_train, X_val, X_test_n]:
            df.fillna(fill_rules, inplace=True)

    # CONT_COLS = [col for col in X_train.columns if col not in [CATS]]

    # X_train_realmlp = X_train.copy()
    # X_val_realmlp = X_val.copy()
    # X_test_realmlp = X_test_n.copy()
    # for df in [X_train_realmlp, X_val_realmlp, X_test_realmlp]:
    #     for c in df.columns:
    #         df[c] = df[c].fillna(0)
    # model_realmlp.fit(X_train_realmlp, y_train,
    #               X_val_realmlp, y_val,
    #                  )
    # realmlp_val_preds = model_realmlp.predict_proba(X_val_realmlp)[:,1]
    # realmlp_oof_preds[val_idx] = realmlp_val_preds
    # realmlp_test_preds = model_realmlp.predict_proba(X_test_realmlp)[:,1]
    
    # realmlp_val_labels = (realmlp_val_preds > pseudo_threshold).astype(int)
    # realmlp_test_labels = (realmlp_test_preds > pseudo_threshold).astype(int)
    # for col in CATS:
    #     X_train[col] = X_train[col].astype(str)
    #     X_val[col] = X_val[col].astype(str)
    #     X_test_n[col] = X_test_n[col].astype(str)
    # for col in INTER:
    #     X_train[col] = X_train[col].astype(str)
    #     X_val[col] = X_val[col].astype(str)
    #     X_test_n[col] = X_test_n[col].astype(str)
        
    # TE = TargetEncoder(
    #     agg_funcs=['mean'], n_folds=5, smooth='auto', drop_original=False, cv_strategy='kfold', handle_unknown='nan', handle_missing='nan',
    #     verbose=1
    # )
    # X_train = TE.fit_transform(X_train, y_train, columns=CATS)
    # X_val = TE.transform(X_val)
    # X_test_n = TE.transform(X_test_n)

    # TE1 = TargetEncoder(
    #     agg_funcs=['mean'], n_folds=5, smooth='auto', drop_original=True, cv_strategy='kfold', handle_unknown='nan', handle_missing='nan',
    #     verbose=1
    # )
    # X_train = TE1.fit_transform(X_train, y_train, columns=INTER)
    # X_val = TE1.transform(X_val)
    # X_test_n = TE1.transform(X_test_n)

    for col in X_train.columns:
        X_train[col] = X_train[col].astype(str).astype('category')
        X_val[col] = X_val[col].astype(str).astype('category')
        X_test_n[col] = X_test_n[col].astype(str).astype('category')
    # for col in CATS:
    #     X_train[col] = X_train[col].astype(str).astype('category')
    #     X_val[col] = X_val[col].astype(str).astype('category')
    #     X_test_n[col] = X_test_n[col].astype(str).astype('category')
    # X_train.drop(columns=INTER, inplace=True)
    # X_val.drop(columns=INTER, inplace=True)
    # X_test_n.drop(columns=INTER, inplace=True)
    # print(X_train.dtypes)
    # print(f'SCORE FOR REALMLP FOLD{i} : {roc_auc_score(y_val, realmlp_val_preds)}')
    # X_train_xgb = pd.concat([X_train, X_val, X_test_n], axis=0, ignore_index=True)
    # y_train_xgb = pd.concat([y_train, pd.Series(realmlp_val_labels), pd.Series(realmlp_test_labels)], axis=0, ignore_index=True)
    print(X_train.shape)
    # param_grid = {'colsample_bytree': 0.2364,
    #               'gamma': 0.034283,
    #               'max_depth': 6,
    #               'reg_alpha': 0.71367,
    #               'reg_lambda': 4.43564,
    #               'subsample': 0.59394}

    # model = xgb.XGBClassifier(**param_grid,
    #                       n_estimators=10000,
    #                       objective='binary:logistic',
    #                       eval_metric='auc',
    #                       learning_rate=0.01,
    #                       early_stopping_rounds=500,
    #                       max_bin=1024,
    #                       random_state=42,
    #                       enable_categorical=True,
    #                       device='cuda',
    #                       n_jobs=-1)
    # cat_params = {
    # 'iterations': 10000,
    # 'learning_rate': 0.03,
    # 'depth': 6,
    # 'l2_leaf_reg': 3,
    # 'bagging_temperature': 1,
    # 'random_strength': 1,
    # 'od_type': 'Iter',
    # 'od_wait': 500,
    # 'eval_metric': 'AUC',
    # 'random_seed': 42,
    # 'task_type': 'GPU', 
    # 'devices': '0',
    # 'verbose': 500
    # }

    # # 4. Train
    # model = cb.CatBoostClassifier(**cat_params)
    # 2. Scale ONLY continuous columns
    # scaler = StandardScaler()
    # X_train[CONT_COLS] = scaler.fit_transform(X_train[CONT_COLS])
    # X_val[CONT_COLS] = scaler.transform(X_val[CONT_COLS])
    # X_test_n[CONT_COLS] = scaler.transform(X_test_n[CONT_COLS])
    

    # train_pool = Pool(
    #     data=X_train, 
    #     label=y_train, 
    #     cat_features=CATS
    # )
    # val_pool = Pool(
    #     data=X_val, 
    #     label=y_val, 
    #     cat_features=CATS
    # )
    model.fit(X_train, y_train,
                X_val, y_val
             )
    preds = model.predict_proba(X_val)[:,1]
    oof_preds[val_idx] = preds
    test_preds += (model.predict_proba(X_test_n)[:, 1] / CONFIG.N_FOLDS)
    print(f'SCORE FOR FOLD{i} : {roc_auc_score(y_val, preds)}')
    
overall_roc_auc = roc_auc_score(y, oof_preds)
print(f'SCORE ACROSS ALL FOLDS : {overall_roc_auc}')

(504000, 67)
Columns classified as continuous: []
Columns classified as categorical: ['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium', 'elevated_bp', 'elevated_chol', 'Age_mean_te', 'Sex_mean_te', 'Chest pain type_mean_te', 'BP_mean_te', 'Cholesterol_mean_te', 'FBS over 120_mean_te', 'EKG results_mean_te', 'Max HR_mean_te', 'Exercise angina_mean_te', 'ST depression_mean_te', 'Slope of ST_mean_te', 'Number of vessels fluro_mean_te', 'Thallium_mean_te', 'Age_median_te', 'Age_std_te', 'Age_count_te', 'Sex_median_te', 'Sex_std_te', 'Sex_count_te', 'Chest pain type_median_te', 'Chest pain type_std_te', 'Chest pain type_count_te', 'BP_median_te', 'BP_std_te', 'BP_count_te', 'Cholesterol_median_te', 'Cholesterol_std_te', 'Cholesterol_count_te', 'FBS over 120_median_te', 'FBS over 120_std_te', 'FBS over 120_count_te', 'EKG results_median_te', 'EKG results_std_

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 1/100: val class_error = 0.112476
Epoch 2/100: val class_error = 0.110389
Epoch 3/100: val class_error = 0.109802
Epoch 4/100: val class_error = 0.109373
Epoch 5/100: val class_error = 0.109635
Epoch 6/100: val class_error = 0.109230
Epoch 7/100: val class_error = 0.109397
Epoch 8/100: val class_error = 0.109270
Epoch 9/100: val class_error = 0.109778
Epoch 10/100: val class_error = 0.109246
Epoch 11/100: val class_error = 0.109611
Epoch 12/100: val class_error = 0.109778
Epoch 13/100: val class_error = 0.109508
Epoch 14/100: val class_error = 0.109468
Epoch 15/100: val class_error = 0.109643
Epoch 16/100: val class_error = 0.109667
Epoch 17/100: val class_error = 0.109714
Epoch 18/100: val class_error = 0.109865
Epoch 19/100: val class_error = 0.110032
Epoch 20/100: val class_error = 0.110008
Epoch 21/100: val class_error = 0.110167
Epoch 22/100: val class_error = 0.110111
Epoch 23/100: val class_error = 0.110214
Epoch 24/100: val class_error = 0.109722
Epoch 25/100: val class_e

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


SCORE FOR FOLD1 : 0.9560689016369956
(504000, 67)
Columns classified as continuous: []
Columns classified as categorical: ['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium', 'elevated_bp', 'elevated_chol', 'Age_mean_te', 'Sex_mean_te', 'Chest pain type_mean_te', 'BP_mean_te', 'Cholesterol_mean_te', 'FBS over 120_mean_te', 'EKG results_mean_te', 'Max HR_mean_te', 'Exercise angina_mean_te', 'ST depression_mean_te', 'Slope of ST_mean_te', 'Number of vessels fluro_mean_te', 'Thallium_mean_te', 'Age_median_te', 'Age_std_te', 'Age_count_te', 'Sex_median_te', 'Sex_std_te', 'Sex_count_te', 'Chest pain type_median_te', 'Chest pain type_std_te', 'Chest pain type_count_te', 'BP_median_te', 'BP_std_te', 'BP_count_te', 'Cholesterol_median_te', 'Cholesterol_std_te', 'Cholesterol_count_te', 'FBS over 120_median_te', 'FBS over 120_std_te', 'FBS over 120_count_te', 'EKG 

...

SCORE FOR FOLD4 : 0.9553812262021634
(504000, 67)
Columns classified as continuous: []
Columns classified as categorical: ['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium', 'elevated_bp', 'elevated_chol', 'Age_mean_te', 'Sex_mean_te', 'Chest pain type_mean_te', 'BP_mean_te', 'Cholesterol_mean_te', 'FBS over 120_mean_te', 'EKG results_mean_te', 'Max HR_mean_te', 'Exercise angina_mean_te', 'ST depression_mean_te', 'Slope of ST_mean_te', 'Number of vessels fluro_mean_te', 'Thallium_mean_te', 'Age_median_te', 'Age_std_te', 'Age_count_te', 'Sex_median_te', 'Sex_std_te', 'Sex_count_te', 'Chest pain type_median_te', 'Chest pain type_std_te', 'Chest pain type_count_te', 'BP_median_te', 'BP_std_te', 'BP_count_te', 'Cholesterol_median_te', 'Cholesterol_std_te', 'Cholesterol_count_te', 'FBS over 120_median_te', 'FBS over 120_std_te', 'FBS over 120_count_te', 'EKG 

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 1/100: val class_error = 0.111452
Epoch 2/100: val class_error = 0.111294
Epoch 3/100: val class_error = 0.110841
Epoch 4/100: val class_error = 0.110786
Epoch 5/100: val class_error = 0.110460
Epoch 6/100: val class_error = 0.110397
Epoch 7/100: val class_error = 0.110310
Epoch 8/100: val class_error = 0.110429
Epoch 9/100: val class_error = 0.110627
Epoch 10/100: val class_error = 0.110524
Epoch 11/100: val class_error = 0.110635
Epoch 12/100: val class_error = 0.110984
Epoch 13/100: val class_error = 0.110944
Epoch 14/100: val class_error = 0.110476
Epoch 15/100: val class_error = 0.110730
Epoch 16/100: val class_error = 0.110468
Epoch 17/100: val class_error = 0.110778
Epoch 18/100: val class_error = 0.110897
Epoch 19/100: val class_error = 0.111008
Epoch 20/100: val class_error = 0.110873
Epoch 21/100: val class_error = 0.110802
Epoch 22/100: val class_error = 0.110944
Epoch 23/100: val class_error = 0.110976
Epoch 24/100: val class_error = 0.110730
Epoch 25/100: val class_e

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


SCORE FOR FOLD5 : 0.956203950374723
SCORE ACROSS ALL FOLDS : 0.9555707004406384


In [27]:
sample_sub['Heart Disease'] = test_preds
sample_sub.to_csv(f'submission_{overall_roc_auc}.csv', index=False)